# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shreyashgol/assignment1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**What one row means:** One row represents one pseudonymized content item on a single report day for a given client (a page-day).
**Tables to use:** `fact_content_daily_performance` joined with `dim_content`.
**Time window:** The month of March 2026 (`month=2026-03`).

In [1]:
import duckdb
import os
import pandas as pd

# Read token from environment (or Colab secrets) so it's not hardcoded
hf_token = os.environ.get('HF_TOKEN', '')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

# Check grain: One row per content_hash_id per report_date
grain_check = con.execute("""
    SELECT content_hash_id, report_date, COUNT(*) as c
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    GROUP BY content_hash_id, report_date
    HAVING c > 1 LIMIT 5
""").fetchall()
print(f"Grain check (expect 0 if grain holds): {len(grain_check)}")

# Check row count and date span
span_check = con.execute("""
    SELECT COUNT(*) as row_count, MIN(report_date) as start_date, MAX(report_date) as end_date
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()
print("\nRow count and date span:")
print(span_check)


HTTPException: HTTP Error: HTTP GET error on 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet' (HTTP 0 Internal Server Error)

## 2. Fields: feature / label / context / excluded

**Target/Proxy:** We want to rank candidates by predicting future opportunity. A proxy is `trend_pct` (future directional movement).
**Features:** `impressions`, `sessions`, `avg_position`, `content_age_days`, `word_count`.
**Context:** `content_hash_id`, `client_hash_id`, `report_date`.
**Deliberately excluded:** `priority_score` (and any other product decisions) because these represent the system's own past decisions, not observable facts from search/analytics.

In [2]:
# Availability check: filter with IS TRUE to show how many rows survive
avail_check = con.execute("""
    SELECT COUNT(*) as valid_rows
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE ga4_data_available IS TRUE
""").df()
print("Rows with GA4 data available (IS TRUE):")
print(avail_check)

print("\n--- Feature Frame (5 features) ---")
print("1. gsc_impressions: knowable at the decision moment — recorded daily by Google Search Console.")
print("2. ga4_sessions: knowable at the decision moment — recorded daily by GA4.")
print("3. gsc_avg_position: knowable at the decision moment — GSC reports it post-day.")
print("4. content_age_days: knowable at the decision moment — derived from static content_created_date.")
print("5. word_count: knowable at the decision moment — an observable property of the published page.")


HTTPException: HTTP Error: HTTP GET error on 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet' (HTTP 0 Internal Server Error)

## 3. Verify it with queries (grain, counts, missing values, windows)

*Note: Grain and counts were verified above. Here we demonstrate the leakage trap.*

In [3]:
# Building a quick slice for the trap
df_features = con.execute("""
    SELECT f.content_hash_id, f.gsc_impressions, f.gsc_avg_position, f.gsc_clicks, 
           c.word_count,
           DATE '2026-03-15' - c.content_created_date AS content_age_days
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet' f
    JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' c
      ON f.content_hash_id = c.content_hash_id
    WHERE c.content_created_date IS NOT NULL
    LIMIT 1000
""").df()
print(f"Feature frame shape: {df_features.shape}")
print(df_features.head())

# TRAP: We add a label-derived column. If we are trying to predict future clicks,
# including the clicks themselves as a feature leaks the answer.
df_features['TRAP_future_clicks'] = df_features['gsc_clicks']
print("\nTrap added: TRAP_future_clicks (perfect copy of gsc_clicks).")
print("This causes massive leakage — any model would score near-perfect by just reading the leaked column.")
print("Removing trap...")
df_features = df_features.drop(columns=['TRAP_future_clicks'])
print("Trap removed. We strictly keep only observable past signals.")


HTTPException: HTTP Error: HTTP GET error on 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet' (HTTP 0 Internal Server Error)

## 4. Data limits

**Limitation:** Unbalanced history. Not every client has tracking installed for the full timeline. Rows prior to their `ga4_data_start` are filled with zeros and `ga4_data_available = FALSE`. This means 'no data', not 'zero traffic'.

In [4]:
# Demonstrate the limitation: check how many rows are missing GA4 tracking completely
limit_check = con.execute("""
    SELECT 
        ga4_data_available, 
        COUNT(*) as count
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    GROUP BY ga4_data_available
""").df()
print("Data Limitation Evidence: A significant portion of rows lack GA4 tracking data completely:\n")
print(limit_check)


HTTPException: HTTP Error: HTTP GET error on 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet' (HTTP 0 Internal Server Error)

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.